## Extend single head attention to multi head

In [18]:
import torch
import torch.nn as nn

In [19]:
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape)

torch.Size([2, 6, 3])


In [20]:
class CausalAttention(nn.Module):
    # context_length is the number of tokens in a sequence to train on
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask",
                             torch.triu(torch.ones(context_length, context_length),
                                        diagonal=1)
                            )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)
        
        attn_scores = queries @ keys.transpose(1, 2)
        # num_tokens is the input token size, masked_fill_ is inplace
        attn_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)
        
        context_vec = attn_weights @ values
        return context_vec

### Stack multiple instances of CausalAttention

In [21]:
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList(\
            [CausalAttention(d_in, d_out, context_length, dropout, qkv_bias)
             for _ in range(num_heads)])

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)  # dim=-1, stack horizontally (side by side)

In [22]:
torch.manual_seed(123)
context_length = batch.shape[1]
d_in, d_out = 3, 2
mha = MultiHeadAttentionWrapper(d_in, d_out, context_length, 0.0, 2)
context_vecs = mha(batch)
print(context_vecs)
print(f"context.vecs.shape: {context_vecs.shape}")

tensor([[[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]],

        [[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]]], grad_fn=<CatBackward0>)
context.vecs.shape: torch.Size([2, 6, 4])


#### Exercise: Return 2 dimensional context vectors instead of 4

In [23]:
torch.manual_seed(123)
context_length = batch.shape[1]
d_in, d_out = 3, 1
mha = MultiHeadAttentionWrapper(d_in, d_out, context_length, 0.0, 2)
context_vecs = mha(batch)
print(context_vecs)
print(f"context.vecs.shape: {context_vecs.shape}")

tensor([[[-0.5740,  0.2216],
         [-0.7320,  0.0155],
         [-0.7774, -0.0546],
         [-0.6979, -0.0817],
         [-0.6538, -0.0957],
         [-0.6424, -0.1065]],

        [[-0.5740,  0.2216],
         [-0.7320,  0.0155],
         [-0.7774, -0.0546],
         [-0.6979, -0.0817],
         [-0.6538, -0.0957],
         [-0.6424, -0.1065]]], grad_fn=<CatBackward0>)
context.vecs.shape: torch.Size([2, 6, 2])


### Implementing multi-head attention with weight splits

#### Breakdown the dimensions

In [93]:
torch.manual_seed(123)

num_heads = 2
d_out = 4
b, num_tokens, d_in = batch.shape
head_dim = d_out // num_heads

W_query = nn.Linear(d_in, d_out, bias=False)
queries = W_query(batch)

W_key = nn.Linear(d_in, d_out, bias=False)
keys = W_key(batch)

W_value = nn.Linear(d_in, d_out, bias=False)
values = W_value(batch)

print(f"Batch shape (x): {batch.shape}")
print(f"W_key shape: [{d_in}, {d_out}]")
print(f"queries shape: {queries.data.shape}")
print(f"keys shape: {keys.data.shape}")

keys = keys.view(b, num_tokens, num_heads, head_dim)
queries = queries.view(b, num_tokens, num_heads, head_dim)
values = values.view(b, num_tokens, num_heads, head_dim)
print(f"\nSplit queries shape: {queries.data.shape}")

queries = queries.transpose(1, 2)
keys = keys.transpose(1, 2)
values = values.transpose(1, 2)
print("\nTranspose queries and keys (1,2):")
print(f"queries shape: {queries.data.shape}")
print(f"keys shape: {keys.data.shape}")
print(f"Split keys.transpose(2, 3) shape: {keys.transpose(2, 3).shape}")

print("\nhead 1:")
print(f"keys[0, 0]:\n{keys[0, 0]}")

print("\nhead 2:")
print(f"keys[0, 1]:\n{keys[0, 1]}")

print(f"\nkeys.transpose(2, 3)[0, 0]:\n{keys.transpose(2, 3)[0, 0]}")

attn_scores = queries @ keys.transpose(2, 3)
print(f"\nhead 1\nattn_scores shape: {attn_scores.data.shape}")
print(f"attn_scores[0, 0]:\n{attn_scores[0, 0]}")

context_length = attn_scores.shape[-1]
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
attn_scores = attn_scores.masked_fill(mask.bool(), -torch.inf)
print(f"\nmasked attn_scores[0, 0]:\n{attn_scores[0, 0]}")

dropout = nn.Dropout(0.5)
attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)
attn_weights = dropout(attn_weights)
print("\nhead 1:")
print(f"dropout attn_scores[0, 0]:\n{attn_weights[0, 0]}")
print("\nhead 2:")
print(f"dropout attn_scores[0, 0]:\n{attn_weights[0, 1]}")

context_vec = (attn_weights @ values).transpose(1, 2)
print("\nhead 1:")
print(f"context_vec[0, :, 0]:\n{context_vec[0, :, 0]}")
print("\nhead 2:")
print(f"context_vec[0, :, 1]:\n{context_vec[0, :, 1]}")

print(f"\ncontext_vec[0]:\n{context_vec[0]}")

context_vec = context_vec.contiguous().view(b, num_tokens, d_out)
print(f"\ncontiguous view context_vec[0]:\n{context_vec[0]}")

Batch shape (x): torch.Size([2, 6, 3])
W_key shape: [3, 4]
queries shape: torch.Size([2, 6, 4])
keys shape: torch.Size([2, 6, 4])

Split queries shape: torch.Size([2, 6, 2, 2])

Transpose queries and keys (1,2):
queries shape: torch.Size([2, 2, 6, 2])
keys shape: torch.Size([2, 2, 6, 2])
Split keys.transpose(2, 3) shape: torch.Size([2, 2, 2, 6])

head 1:
keys[0, 0]:
tensor([[-0.4519,  0.2216],
        [-0.7142, -0.1961],
        [-0.7127, -0.1971],
        [-0.3809, -0.1557],
        [-0.4861, -0.1597],
        [-0.4213, -0.1501]], grad_fn=<SelectBackward0>)

head 2:
keys[0, 1]:
tensor([[ 0.3326,  0.5659],
        [ 0.3558,  0.5643],
        [ 0.3412,  0.5522],
        [ 0.2123,  0.2991],
        [-0.0177,  0.1780],
        [ 0.3660,  0.4382]], grad_fn=<SelectBackward0>)

keys.transpose(2, 3)[0, 0]:
tensor([[-0.4519, -0.7142, -0.7127, -0.3809, -0.4861, -0.4213],
        [ 0.2216, -0.1961, -0.1971, -0.1557, -0.1597, -0.1501]],
       grad_fn=<SelectBackward0>)

head 1
attn_scores shape:

#### Implement the class

In [70]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible bu num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        # split to dims of `batch size` x `num tokens` x `num heads` x `head dim`
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)

        queries = queries.transpose(1, 2)
        keys = keys.transpose(1, 2)
        values = values.transpose(1, 2)
        
        attn_scores = queries @ keys.transpose(2, 3)
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = (attn_weights @ values).transpose(1, 2)
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)
        return context_vec

In [71]:
torch.manual_seed(123)
batch_size, context_length, d_in = batch.shape
d_out = 2
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)
context_vecs = mha(batch)
print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]],

        [[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]]], grad_fn=<ViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])
